# Brent · `appeal_decision` (web: "Appeal decision")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `appeal_decision` and `appeal_decision_date` for every Brent application (paged
with `search_after`, sorted by `id`, because there are more than 10,000). Count for
`appeal_decision`:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

Then count the four combinations of the two fields being filled or not.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "appeal_decision", "appeal_decision_date"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "appeal_decision", "appeal_decision_date"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["appeal_decision"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,35647,96.47
empty or blank,0,0.00
filled,1305,3.53
total,36952,100.00


In [4]:
has_decision = df["appeal_decision"].notna() & (df["appeal_decision"].astype(str).str.strip() != "")
has_date = df["appeal_decision_date"].notna() & (df["appeal_decision_date"].astype(str).str.strip() != "")
pd.crosstab(has_decision.rename("has appeal_decision"), has_date.rename("has appeal_decision_date"), margins=True)

has appeal_decision_date,False,True,All
has appeal_decision,,,
False,35647,0,35647
True,1,1304,1305
All,35648,1304,36952


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| `appeal_decision` | Count | Share |
|---|---|---|
| null | 35,647 | 96.47% |
| empty or blank | 0 | 0% |
| filled | 1,305 | 3.53% |

| | no `appeal_decision_date` | has `appeal_decision_date` |
|---|---|---|
| no `appeal_decision` | 35,647 | 0 |
| has `appeal_decision` | 1 | 1,304 |

- **About 3.5% have an appeal decision.** Null is the normal case: most applications are
  never appealed.
- **The two fields go together:** every appeal decision date has an appeal decision, and
  all but 1 appeal decision have a date.

## Step 2 · The record with an appeal decision but no appeal decision date

Fetch it with the web columns plus the other appeal fields and `bo_system`.

In [5]:
COLUMNS = [
    "id", "lpa_app_no", "valid_date", "status", "decision", "decision_date", "application_type",
    "description", "site_name", "decision_target_date", "appeal_decision", "appeal_decision_date",
    "appeal_start_date", "url_planning_app", "bo_system", "last_updated",
]
ids = df.loc[has_decision & ~has_date, "id"].tolist()
hits = search({"size": 10, "query": {"bool": {"filter": [brent_filter(), {"ids": {"values": ids}}]}},
               "_source": COLUMNS})["hits"]["hits"]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([h["_source"] for h in hits]).reindex(columns=COLUMNS).T

,0
id,Brent-55_0100
lpa_app_no,55/0100
valid_date,20/05/2026
status,Application Under Consideration
decision,Approved
decision_date,None
application_type,All Other
description,Test application record for system monitoring only. Do not delete record. TEST
site_name,"4 Alderside Apartments, 35 Salusbury Road, London, NW6 6BF"
decision_target_date,02/06/2026


### Findings · Step 2

Live API, 2026-11-10:

- **The one mismatch is the test record `Brent-55_0100`** ("Test application record for
  system monitoring only"), already dropped at load by `drop_test_records`.
- Its fields contradict each other: status Under Consideration, decision Approved, appeal
  Allowed, appeal start 12/02/2026 before its valid date 20/05/2026.
- **After cleaning, the two fields always go together:** a record has both an
  `appeal_decision` and an `appeal_decision_date`, or neither.

## Step 3 · Do the values match the schema type?

The schema gives `appeal_decision` the type `string`. Count the Python type of every value
in `df`: `str` = JSON string, `int` / `float` = JSON number, `NoneType` = null.

In [6]:
types = df["appeal_decision"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["appeal_decision"].value_counts().head(30).to_frame("count")

,count
appeal_decision,
NoneType,35647
str,1305


values that are not strings: 0


,count
appeal_decision,


### Findings · Step 3

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 1,305 |
| null | 35,647 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says. No type mismatch.

## Step 4 · All `appeal_decision` values

Count every value in `df` (exact text), and compare with the schema's list.
`shown_as` uses `repr()` so hidden characters are visible.

In [7]:
# From planninglondondatahub_public_technical_schemav2.1.xlsx, sheet "Field Values",
# index "applications", field "appeal_decision" (extracted 2026-11-10).
SCHEMA_APPEAL_DECISION = ["Allowed", "Dismissed", "Appeal In Progress", "Quashed", "Split Decision", "Withdrawn", "Appeal Closed"]

counts = df["appeal_decision"].value_counts(dropna=False).to_frame("count").reset_index()
counts["shown_as"] = counts["appeal_decision"].map(repr)
counts["in_schema"] = counts["appeal_decision"].isin(SCHEMA_APPEAL_DECISION)
print("schema values never seen:", [v for v in SCHEMA_APPEAL_DECISION if v not in set(df["appeal_decision"].dropna())])
counts

schema values never seen: ['Appeal In Progress', 'Quashed', 'Appeal Closed']


,appeal_decision,count,shown_as,in_schema
0,NaN,35647,nan,False
1,Dismissed,729,'Dismissed',True
2,Allowed,493,'Allowed',True
3,Withdrawn,43,'Withdrawn',True
4,Split Decision,24,'Split Decision',True
5,ZZZ,16,'ZZZ',False


### Findings · Step 4

Live API, 2026-11-10:

| `appeal_decision` | Count | In schema |
|---|---|---|
| Dismissed | 729 | yes |
| Allowed | 493 | yes |
| Withdrawn | 43 | yes |
| Split Decision | 24 | yes |
| ZZZ | 16 | no |
| null | 35,647 | — |

- **Clean apart from `ZZZ`:** no case or whitespace variants. `ZZZ` is the same 16 records
  as status `ZZZ` (open question 2).
- **Never seen:** `Appeal In Progress`, `Quashed`, `Appeal Closed`. So appeals still
  running show only in `status` (Appeal Received / Appeal In Progress), not here.
- **More than `status` shows:** status says Allowed 242 and Dismissed 719, but
  `appeal_decision` has Allowed 493 and Dismissed 729. `status` is overwritten as the
  application moves on (e.g. to Completed after an allowed appeal), so appeal outcomes
  should be counted from `appeal_decision`.

## Step 5 · `status` against `appeal_decision`

Fetch `status` for every record and count it against `appeal_decision`, to see what the
status says now for each appeal outcome, and whether any status says Allowed or Dismissed
without a matching appeal decision. Nulls are shown as `(none)`.

In [8]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "status", "appeal_decision"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

sa = pd.DataFrame(rows).reindex(columns=["id", "status", "appeal_decision"]).fillna("(none)")

# Only records that have an appeal decision, or a status that is an appeal outcome.
APPEAL_STATUS = ["Allowed", "Dismissed", "Split Decision", "Appeal Received", "Appeal In Progress", "Appeal in Progress", "ZZZ"]
subset = sa[(sa["appeal_decision"] != "(none)") | sa["status"].isin(APPEAL_STATUS)]
pd.crosstab(subset["status"], subset["appeal_decision"], margins=True)

appeal_decision,(none),Allowed,Dismissed,Split Decision,Withdrawn,ZZZ,All
status,,,,,,,
Allowed,0,242,0,0,0,0,242
Appeal In Progress,1,0,0,0,0,0,1
Appeal Received,60,0,0,0,0,0,60
Appeal in Progress,2,0,0,0,0,0,2
Application Under Consideration,0,1,0,0,0,0,1
Closed,0,8,10,1,2,0,21
Commenced,0,2,0,0,0,0,2
Completed,0,155,0,1,0,0,156
Dismissed,0,0,719,0,0,0,719


### Findings · Step 5

Live API, 2026-11-10, what `status` says now for each `appeal_decision`:

| `appeal_decision` | `status` now |
|---|---|
| Allowed (493) | Allowed 242, Completed 155, Lapsed 52 (incl. 2 `lapsed`), Superseded 33, Closed 8, Commenced 2, Under Consideration 1 (the test record `Brent-55_0100`) |
| Dismissed (729) | Dismissed 719, Closed 10 |
| Split Decision (24) | Split Decision 22, Closed 1, Completed 1 |
| Withdrawn (43) | Withdrawn 30, Refused 11, Closed 2 |
| ZZZ (16) | ZZZ 16 |
| none | Appeal Received 60, Appeal In Progress 3 (incl. 2 `Appeal in Progress`) |

- **Confirmed: `status` moved on after the appeal.** About half of the allowed appeals are
  now Completed, Lapsed, Superseded, Closed or Commenced. A dismissed appeal leaves nothing
  to build, so its status almost never changes (only 10 are now Closed).
- **No contradictions:** every status Allowed has appeal Allowed, every status Dismissed
  has appeal Dismissed, and every status Split Decision has appeal Split Decision.
- **The 22 `Split Decision` statuses are appeal outcomes**, not council decisions.
- **A withdrawn appeal leaves the council's decision standing:** 11 now show status Refused.
- **Running appeals have no `appeal_decision` yet:** Appeal Received 60, Appeal In
  Progress 3.